# Four-site eigenvalues and quadratic baseline

Evaluate Eq. (13) and the direct-diagonalization baseline for Fig. 5 and Sec. V D. The original primary calculation used interpolation over physical couplings; the independent verification used uniform ratios. This notebook explicitly uses the latter, so its metrics are not the original primary checkpoint's validation metrics.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiments.paper import ROOT
print('Repository:', ROOT)

In [ ]:
from experiments.paper import exact_four, published_predictions, basis, make_dataset, metrics
config = json.loads((ROOT/'configs/paper/four_site_verification.json').read_text())
x,y,train,validation=make_dataset(config)
coeff=np.linalg.lstsq(basis(x)[train],y[train],rcond=None)[0]
pd.DataFrame({
    'OLS quadratic, validation': metrics(basis(x)[validation]@coeff,y[validation]),
    'Rounded Eq. (13), validation': metrics(published_predictions(x[validation]),y[validation]),
}).T[['rmse','max_abs_error']]

**Reproduction discrepancy:** the stated uniform-ratio protocol gives OLS RMSE about 0.01823, not the manuscript’s 0.00702. Rounded Eq. (13) gives about 0.04139 on this split. The physical-coupling sampling gives different errors (about 0.00683 for OLS). These protocols are kept separate; see the numerical discrepancy section of the reproduction guide. The original independent verification code/data are needed to resolve the mismatch.

In [ ]:
r=np.linspace(0,3,601)[:,None]
exact=exact_four(r)
np.testing.assert_allclose(exact+exact[:,::-1],-2,atol=2e-14)
fig,ax=plt.subplots(figsize=(8,6))
ax.plot(r,exact,color='steelblue')
ax.plot(r,published_predictions(r),'--',color='firebrick')
ax.plot([],[],color='steelblue',label='Exact diagonalization')
ax.plot([],[],'--',color='firebrick',label='Rounded Eq. (13)')
ax.set(xlabel='r = deltaJ/J',ylabel='E/J');ax.legend()
plt.show()
print('OLS coefficients: constant, linear, quadratic')
display(pd.DataFrame(coeff.T,columns=['constant','linear','quadratic'],index=range(1,9)))

## Sampling and training protocols

```bash
# Primary: physical couplings sampled separately, 200x200 tracked interpolation grid.
python -m experiments.paper --config configs/paper/four_site.json --train
# Verification: direct diagonalization, uniform r in [0,3].
python -m experiments.paper --config configs/paper/four_site_verification.json --train
```
Both use `(1,10,8)`, 5000 epochs, Adam at 0.001, MSE and batch size 8000. Omit `--train` to evaluate the baselines and rounded formulas only. All configurations, datasets and split indices are saved. The supplied repository did not include the independently written historical verification implementation or its checkpoint; the new runner uses the shared repository ASN and labels its output as a fresh run.